# Taylor-Green Vortex (2D, weakly compressible)

A periodic box of fluid stamped with the Taylor-Green velocity field: a grid of
counter-rotating vortices that, in the continuum, decays without changing shape.
That is what makes it the calibration case of this family -- it is the only
setup here whose dissipation has a closed-form answer, so the scheme's actual
viscosity can be *measured* rather than assumed:

$$
E_k(t) = E_k(0)\, e^{-4 \nu k^2 t}, \qquad E_k(0) = \tfrac14 \rho_0 u_{mag}^2 L^2
$$

`analyticDecayRate`, `analyticKineticEnergy` and `effectiveViscosity` in
`warpSPH.cases.tgvWeaklyCompressible` are those three formulas; the notebook
calls them rather than restating them.

Two mechanisms are worth reading before changing numbers:

- **The measured decay is about half the prescribed one.** The diffusion
  operator carries the classic Monaghan switch, which applies viscosity only to
  particle pairs that are *approaching*. At any instant that is roughly half the
  pairs, so roughly half the dissipation arrives. It is not a bug to be tuned
  away -- turning the switch off recovers the analytic rate and costs stability
  elsewhere -- but it does mean `--nu 0.01` is a request, not a measurement.
- **`nu` and `alpha` are one knob written two ways.** The physical viscosity and
  the deltaSPH artificial-viscosity coefficient are related by
  $\nu = \alpha c_0 h / (2(n+2))$, so once the sound speed and the support radius
  exist, each implies the other (`viscosityScales`). This matters because
  $\alpha \gtrsim 0.01$ is a *stability* floor: at a given resolution it sets the
  smallest viscosity, and therefore the largest Reynolds number, this
  discretisation can carry. Ask for less and the scheme's own numerical
  dissipation answers instead, which the sweep at the end of this notebook shows
  directly.

![](outputs/05-taylorGreenVortex.gif)

## Every knob, and what it does

The parameters cell below is the whole command line of `05-taylor-green-vortex.py`
written out: `CaseSpec` fields first, then `tgvWeaklyCompressibleCase.params` --
the case's own physics knobs, each of which is also a `--flag`. Anything not
named there keeps the value in `tgvWeaklyCompressibleCase.defaults`/`.params`.

**Discretisation, time stepping and output** (`CaseSpec` fields, shared by every case)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `256` | particles across the domain; the spacing is `dx = L / nx` |
| `dim` | `2` | this case is 2D; the decay law above is the 2D one |
| `L` | `2 * pi` | side of the (periodic) box, chosen so `k` is an integer number of periods |
| `n_h` | `4.0` | particles per support radius, i.e. how smooth the kernel is |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | time integrator |
| `scheme` | `deltaSPH` | the solver itself |
| `tLimit` | `2.0` | simulated end time; the loop runs `tLimit / dt` steps |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `60` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `storeInterval` | `False`, `'states'`, `500` | HDF5 export; off here |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `k` | `2` | vortices across the box; the field's wavenumber is `k / 2`, and an even `k` gets a quarter-period phase shift so the vortex centres do not land on the domain edge |
| `uMag` | `1.0` | peak speed of the initial field, the `u_mag` of the energy formula above |
| `shuffleIters` | `128` | relaxation iterations applied to the lattice before the run; a perfectly regular lattice is an *unstable* SPH equilibrium and decays into lattice noise |
| `jitter` | `1.0` | how hard that relaxation is allowed to push particles |
| `inviscid` | `False` | `False` uses the physical viscosity `nu`; `True` leaves the artificial viscosity `alpha` as the only dissipation |
| `nu` | `0.01` | kinematic viscosity, the `nu` of the decay law -- what this run *asks* for |
| `alpha` | `0.01` | artificial-viscosity coefficient; unused while `inviscid=False`, but it is what `nu` converts *to*, and `0.01` is the stability floor |
| `rho0` | `1.0` | rest density |
| `targetDt` | `0.001` | the timestep the run asks for; the sound speed is then chosen to make it the acoustic CFL limit |
| `freeSurface` | `False` | surface detection; this case is a closed periodic box with no surface |
| `band` | `0` | particle layers of boundary padding around the domain; none, the box is periodic |
| `markerSize` | `4` | plot only: particle marker size |

**Three things this family does differently from the compressible notebooks**
(they will bite if `../compressible/08-hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `tgvWeaklyCompressibleCase.initialConditions(ctx, system)`.
   That is where `setupWeaklyCompressibleTimestep` picks the sound speed and
   `config.dt` *together* from `targetDt` -- weakly compressible SPH is free to
   choose its own stiffness, so the timestep is fixed first and `c0` follows
   from the acoustic CFL. Skip it and `config.dt` stays `None`, and the vortex
   field itself is never stamped onto the particles.
2. **The loop is `range(nSteps)`.** No case in this family has a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` on `VELOCITY_DENSITY_FIELDS`
   directly rather than `tgvWeaklyCompressibleCase.setupPlot`/`updatePlot`, which go
   through `openWindow`/`pumpEvents` and do not live-update inside a Jupyter cell
   in this environment -- `08-hydrostatic.ipynb` explains that in full.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.tgvWeaklyCompressible import (MIN_STABLE_ALPHA, analyticDecayRate,
                                                 analyticKineticEnergy, effectiveViscosity,
                                                 tgvWeaklyCompressibleCase, viscosityScales,
                                                 wavenumber)
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.cases.weaklyCompressible import VELOCITY_DENSITY_FIELDS
from warpSPH.runner import CaseSpec, RunResult, buildContext, encodeFrames, run
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `05-taylor-green-vortex.py`, made explicit and editable here -- the table in
# the intro cell says what each one does. `tgvWeaklyCompressibleCase.defaults`/
# `.params` are the same values the CLI script starts from.
spec = CaseSpec(caseName=tgvWeaklyCompressibleCase.name, scheme=tgvWeaklyCompressibleCase.scheme,
                params=dict(tgvWeaklyCompressibleCase.params)) \
    .merged(**tgvWeaklyCompressibleCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=256,
    dim=2,
    L=2 * np.pi,

    # --- time stepping ---------------------------------------------------
    tLimit=2.0,

    # --- output --------------------------------------------------------------
    caseName='05-taylorGreenVortex',
    plot=True, show=True, plotInterval=60,
    store=False,

    # --- the vortex's own knobs ----------------------------------------------
    params=dict(
        # the field: k vortices across the box, peak speed uMag
        k=2, uMag=1.0,
        # the lattice: a regular one is an unstable SPH equilibrium, so relax it
        shuffleIters=128, jitter=1.0,
        # the dissipation: physical viscosity here, so `alpha` follows from `nu`
        inviscid=False, nu=0.01, alpha=MIN_STABLE_ALPHA,
        # the fluid
        rho0=1.0, targetDt=0.001, freeSurface=False, band=0,
        markerSize=4,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`tgvWeaklyCompressibleCase.buildSystem`), not re-derived here.
#
# `initialConditions` is the call the compressible notebooks do not have: the
# Taylor-Green field is stamped there, and it is where the sound speed and
# `config.dt` are chosen together from `targetDt`, so skipping it leaves
# `config.dt` unset.
ctx = buildContext(tgvWeaklyCompressibleCase, spec)
tgvWeaklyCompressibleCase.configureScheme(ctx)
system = tgvWeaklyCompressibleCase.buildSystem(ctx)
tgvWeaklyCompressibleCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

print(f'dt = {float(ctx.config.dt):.3e}, '
      f'c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles')

# The cheapest check that the lattice, the masses and the shuffle all came out
# right: the sampled particles should carry the continuum energy of the field
# they were stamped with, to within the discretisation error.
ke0 = (0.5 * system.state.masses * (system.state.velocities ** 2).sum(dim=-1)).sum().item()
print(f'E_k(0) = {ke0:.6g} sampled vs {analyticKineticEnergy(ctx):.6g} analytic '
      f'({abs(ke0 / analyticKineticEnergy(ctx) - 1) * 100:.2f}% off)')

In [ ]:
# What was actually built. The other notebooks in this family preview
# `ctx.scratch['regions']` here, but this case has no regions to preview: it
# fills the whole periodic box through `setupBasicWeaklyCompressibleInitialState`
# rather than through SDFs, so what is worth looking at instead is the *field* --
# `k` vortices per side, their centres in the interior rather than on the domain
# edge (that is the phase shift an even `k` gets), and the box running [0, L]^2
# rather than the symmetric box the shared configure block builds by default.
figure, axis = plt.subplots(1, 1, figsize=(5.5, 5), squeeze=False)
domain = ctx.config.domain
positions = system.state.positions.detach().cpu().numpy()
velocities = system.state.velocities.detach().cpu().numpy()

scatter = axis[0, 0].scatter(positions[:, 0], positions[:, 1],
                             c=np.linalg.norm(velocities, axis=-1), s=1, cmap='viridis')
stride = max(1, len(positions) // 400)
axis[0, 0].quiver(positions[::stride, 0], positions[::stride, 1],
                  velocities[::stride, 0], velocities[::stride, 1],
                  color='white', alpha=0.6, scale=30)
figure.colorbar(scatter, ax=axis[0, 0], label='|u|')
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f"k = {spec.param('k')} on "
                     f"[{domain.min[0]:.3g}, {domain.max[0]:.3g}]^2, "
                     f'{len(positions)} particles')
figure.tight_layout()

## What viscosity is this run actually asking for?

The deltaSPH artificial viscosity (Marrone et al. 2012, following Antuono et al.
2009) contributes

$$
\alpha h c_0 \rho_0 \sum_j \Pi_{ij} \nabla_i W_{ij}\, dV_j
$$

while the laminar viscosity (Sun et al. 2016, with $\mu = \nu \rho$) contributes

$$
\mu\, 2(n+2) \sum_j \Pi_{ij} \nabla_i W_{ij}\, dV_j
$$

so the two are the same term with a different coefficient, and equating them,

$$
\alpha h c_0 \rho_0 = \nu \rho\, 2(n+2)
\;\Longrightarrow\;
\alpha = \frac{2\nu(n+2)}{c_0 h}\frac{\rho}{\rho_0};\quad
\nu = \frac{\alpha c_0 h}{2(n+2)}\frac{\rho_0}{\rho}
$$

For a nearly incompressible flow ($Ma < 0.1$, which is what the weakly
compressible sound speed buys) $\rho \approx \rho_0$ and the density ratio drops
out. That is `nuToAlpha`/`alphaToNu`, and it is what lets *either* knob be
converted into a Reynolds number.

The catch is empirical: below $\alpha \approx 0.01$ runs stop being reliably
stable. At a fixed $c_0$ and $h$ that is a floor on $\nu$, and therefore a
**ceiling on the Reynolds number this discretisation can represent** -- ask for
less viscosity than that and you do not get it, you get whatever the scheme's
own numerical dissipation happens to be.

In [ ]:
# `viscosityScales` is the conversion above, applied to whichever knob this run
# was actually given (`nu` here, `alpha` under --inviscid). It needs the built
# system, not just the spec: c0 and the mean support radius only exist once the
# particles do.
scales = viscosityScales(ctx, system)
print(f"c0 = {scales['c0']:.4g}, mean h = {scales['h']:.4g}")
print(f"nu = {scales['nu']:.4g}  <->  alpha = {scales['alpha']:.4g}")
print(f"Re = {scales['Re']:.4g}  (u = {spec.param('uMag')}, length scale L/2 = {spec.L / 2:.4g})")
print(f"floor: alpha >= {MIN_STABLE_ALPHA} means nu >= {scales['nuLimit']:.4g}, "
      f"i.e. Re <= {scales['ReLimit']:.4g}")
if scales['alpha'] < MIN_STABLE_ALPHA:
    print(f"\n!! alpha = {scales['alpha']:.4g} is below the {MIN_STABLE_ALPHA} floor: this run is "
          f"asking for less dissipation than the discretisation can deliver, so the measured\n"
          f"   decay rate will be set by numerical dissipation rather than by nu. "
          f"Raise nx (which lowers h) or nu to fix it.")

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(VELOCITY_DENSITY_FIELDS), not
# tgvWeaklyCompressibleCase.setupPlot -- see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, VELOCITY_DENSITY_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = tgvWeaklyCompressibleCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData,
                              extraFields=tgvWeaklyCompressibleCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

# `weaklyCompressibleDiagnostics` already records `kineticEnergy` every step,
# which is the whole measurement this case exists for -- so the decay fit below
# reads the trajectory rather than accumulating a second list.
trajectory = [dict(tgvWeaklyCompressibleCase.diagnostics(ctx, runningState), step=-1, t=0.0)]
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = tgvWeaklyCompressibleCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, VELOCITY_DENSITY_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                   schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                   extraFields=tgvWeaklyCompressibleCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## Did it decay at the rate it was asked to?

`effectiveViscosity` fits $\nu_{eff}$ from $\log E_k(t)$, which is a one-line
least squares once the trajectory exists -- and it takes a `RunResult`, so the
unrolled loop above is wrapped in one rather than the fit being rewritten here.

Expect $\nu_{eff} \approx \nu / 2$, for the reason in the intro: the Monaghan
switch dissipates only on approaching pairs. The decay stays a clean exponential
either way, which is what the log-scale panel is for -- a curve that bends is
reporting something else (a resolution problem, or a run below the $\alpha$
floor), not a different viscosity.

In [ ]:
result = RunResult(ctx=ctx, state=runningState, trajectory=trajectory)
nuEff = effectiveViscosity(result)
nu = spec.param('nu')
print(f'nu asked for  : {nu:.6g}  (decay rate {analyticDecayRate(ctx):.6g})')
print(f'nu measured   : {nuEff:.6g}  (decay rate {4 * nuEff * wavenumber(ctx) ** 2:.6g})')
print(f'ratio         : {nuEff / nu:.3f}')

figure, axis = plt.subplots(1, 2, figsize=(11, 4))
t = result.series('t')
energy = result.series('kineticEnergy')
k = wavenumber(ctx)

axis[0].plot(t, energy, label='measured')
axis[0].plot(t, energy[0] * np.exp(-4 * nu * k ** 2 * t), ls='--',
             label=fr'asked for ($\nu$ = {nu:.4g})')
axis[0].plot(t, energy[0] * np.exp(-4 * nuEff * k ** 2 * t), ls=':',
             label=fr'fitted ($\nu_{{eff}}$ = {nuEff:.4g})')
axis[0].axhline(analyticKineticEnergy(ctx), color='black', lw=0.8, ls='-.',
                label='continuum $E_k(0)$')
axis[0].set_yscale('log')
axis[0].set_xlabel('t'); axis[0].set_ylabel('$E_k$'); axis[0].legend()

# The weakly compressible health check: the whole scheme rests on the fluid
# staying within about a percent of rho0.
axis[1].plot(t, result.series('maxDensity'), label='max')
axis[1].plot(t, result.series('minDensity'), label='min')
axis[1].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[1].set_xlabel('t'); axis[1].set_ylabel(r'$\rho$'); axis[1].legend()
figure.tight_layout()

## How does the measured viscosity track the prescribed one?

The fit above is one point of a curve. Sweeping $\nu$ over decades and refitting
each run is the actual calibration, and it is where the $\alpha \geq 0.01$ floor
stops being a rule of thumb and becomes visible: above the floor $\nu_{eff}$
tracks $\nu$ at the constant ratio the Monaghan switch implies; below it, the
ratio blows up, because the scheme is delivering its own numerical dissipation
rather than the viscosity that was requested.

Each point is a full `run()` of the same case -- the runner's own loop, not a
copy of the one above -- so the sweep is a `for` loop over `run()` plus
`effectiveViscosity`, and it costs a few lines rather than a second step loop.
It runs at a **coarser `nx` and a shorter `tLimit`** than the main run above so
that eight of them fit in a couple of minutes; the ratio itself is
resolution-dependent (`h` is in the conversion), so re-read the $\alpha$ column
rather than carrying a number over from the run above.

In [ ]:
sweepSpec = spec.merged(nx=128, tLimit=1.0, plot=False, store=False)
sweepNus = np.logspace(-1, -4, 8)

sweep = []
for nu in tqdm(sweepNus, desc='nu sweep'):
    sweepResult = run(tgvWeaklyCompressibleCase,
                      sweepSpec.merged(params=dict(inviscid=False, nu=float(nu))),
                      quiet=True, progress=False)
    scalesOf = viscosityScales(sweepResult.ctx, sweepResult.state)
    sweep.append(dict(nu=float(nu), alpha=scalesOf['alpha'],
                      nuEff=effectiveViscosity(sweepResult),
                      diverged=sweepResult.diverged))
    print(f"nu = {sweep[-1]['nu']:.4g}  alpha = {sweep[-1]['alpha']:.4g}  "
          f"nu_eff = {sweep[-1]['nuEff']:.4g}  ratio = {sweep[-1]['nuEff'] / nu:.3g}"
          + ('  DIVERGED' if sweep[-1]['diverged'] else ''))

In [ ]:
figure, axis = plt.subplots(1, 2, figsize=(12, 4.5))
nus = np.array([row['nu'] for row in sweep])
nuEffs = np.array([row['nuEff'] for row in sweep])
alphas = np.array([row['alpha'] for row in sweep])
# The nu at which alpha hits the stability floor, at this sweep's resolution.
nuFloor = float(np.interp(MIN_STABLE_ALPHA, alphas[::-1], nus[::-1]))

for ax in axis:
    ax.axvspan(nus.min(), nuFloor, color='red', alpha=0.07)
    ax.axvline(nuFloor, color='red', lw=0.8, ls='--',
               label=fr'$\alpha$ = {MIN_STABLE_ALPHA} at $\nu$ = {nuFloor:.3g}')
    ax.set_xscale('log')
    ax.set_xlabel(r'prescribed $\nu$')

axis[0].loglog(nus, nuEffs, marker='o', label=r'measured $\nu_{eff}$')
axis[0].loglog(nus, nus, color='black', lw=0.8, ls=':', label=r'$\nu_{eff} = \nu$')
axis[0].set_ylabel(r'measured $\nu_{eff}$'); axis[0].legend()

axis[1].semilogx(nus, nuEffs / nus, marker='o')
axis[1].axhline(0.5, color='black', lw=0.8, ls=':', label='half (Monaghan switch)')
axis[1].set_ylabel(r'$\nu_{eff} / \nu$'); axis[1].set_yscale('log'); axis[1].legend()
figure.tight_layout()